In [2]:
from pathlib import Path
import csv

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

BARRIER_THRESHOLD_EV = 0.8

# ============================================================
# 1. Input CSV
# ============================================================
# Put energy_001.csv in the same directory as this script.
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter / IPython
    BASE_DIR = Path.cwd()

CSV_FILE = BASE_DIR / "energy_011_12.csv"

print("Working directory :", BASE_DIR)
print("CSV file          :", CSV_FILE)


Working directory : /home/sowon-desktop/ligand-project/EnergeticSpanModel/rNets_diagram/plotly
CSV file          : /home/sowon-desktop/ligand-project/EnergeticSpanModel/rNets_diagram/plotly/energy_011_12.csv


In [3]:
def load_states(csv_file):
    """
    Read:
        path,state,coordinate,relative_E

    O, C, X are parsed automatically from state labels such as S211:
        S211 -> O=2, C=1, X=1

    Rows with blank relative_E are skipped.
    """
    data = []

    with open(csv_file, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)

        required = {"state", "coordinate", "relative_E"}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(
                f"Missing required CSV columns: {sorted(missing)}\n"
                f"Required columns: state, coordinate, relative_E"
            )

        for row in reader:
            state = row["state"].strip()
            energy_text = row["relative_E"].strip()

            # Allow incomplete states, e.g. S002 with no energy yet.
            if not state or not energy_text:
                continue

            if len(state) != 4 or not state.startswith("S") or not state[1:].isdigit():
                raise ValueError(
                    f"State '{state}' must have the form S(O,C,X), e.g. S211."
                )

            O, C, X = map(int, state[1:])

            data.append(
                {
                    "path": row.get("path", "").strip(),
                    "state": state,
                    "coordinate": float(row["coordinate"]),
                    "E0": float(energy_text),
                    "O": O,
                    "C": C,
                    "X": X,
                }
            )

    return data


data = load_states(CSV_FILE)
smap = {s["state"]: s for s in data}

In [4]:
# ============================================================
# 2. Reaction network
# ============================================================
# OL : O + 1
# C  : C + 1
# SR : X + 1
# OX : O + 1 and C + 1
#
# Edges are generated automatically from the loaded states,
# so you do not need to manually list every transition.
# ============================================================
def classify_transition(a, b):
    dO = b["O"] - a["O"]
    dC = b["C"] - a["C"]
    dX = b["X"] - a["X"]

    if (dO, dC, dX) == (1, 0, 0):
        return "OL"
    if (dO, dC, dX) == (0, 1, 0):
        return "C"
    if (dO, dC, dX) == (0, 0, 1):
        return "SR"
    if (dO, dC, dX) == (1, 1, 0):
        return "OX"

    return None


def build_edges(states):
    edges = []

    for a in states:
        for b in states:
            if b["coordinate"] <= a["coordinate"]:
                continue

            transition = classify_transition(a, b)
            if transition is None:
                continue

            # Coordinate convention:
            # OL, C, SR advance by 1;
            # OX increments two counters simultaneously and advances by 2.
            expected_step = 2 if transition == "OX" else 1

            if abs((b["coordinate"] - a["coordinate"]) - expected_step) < 1e-9:
                edges.append((a["state"], b["state"], transition))

    return edges


edges = build_edges(data)

COLOR = {
    "OL": "#2563eb",
    "C":  "#9333ea",
    "SR": "#16a34a",
    "OX": "#dc2626",
}

In [5]:

# ============================================================
# 3. Chemical-potential correction
# ============================================================
# User-defined corrections:
#
# Olation:
#     - Δμ_Mn(OH)2
#
# Condensation:
#     + Δμ_H2O
#
# Oxidation / self-redox:
#     + 1/2 Δμ_H2O
#
# Oxolation:
#     - Δμ_Mn(OH)2 + Δμ_H2O
#
# Because state S(O,C,X) stores cumulative counts:
#
# G = E0 - O*Δμ_Mn(OH)2 + (C + X/2)*Δμ_H2O
# ============================================================
def corrected_energy(s, mu_mn, mu_h2o):
    return (
        s["E0"]
        - s["O"] * mu_mn
        + (s["C"] + 0.5 * s["X"]) * mu_h2o
    )

def get_reachable_edges(mu_mn=0.0, mu_h2o=0.0):
    # 1. 우선 ΔE_step <= 1 eV인 edge만 남김
    allowed_edges = []

    for a_name, b_name, transition in edges:
        a = smap[a_name]
        b = smap[b_name]

        Ea = corrected_energy(a, mu_mn, mu_h2o)
        Eb = corrected_energy(b, mu_mn, mu_h2o)

        dE = Eb - Ea

        if dE <= BARRIER_THRESHOLD_EV:
            allowed_edges.append(
                (a_name, b_name, transition)
            )

    # 2. S000에서 실제로 도달 가능한 state만 탐색
    reachable = {"S000"}
    reachable_edges = []

    changed = True

    while changed:
        changed = False

        for a_name, b_name, transition in allowed_edges:

            if a_name in reachable and b_name not in reachable:
                reachable.add(b_name)
                changed = True

            if a_name in reachable:
                edge = (a_name, b_name, transition)

                if edge not in reachable_edges:
                    reachable_edges.append(edge)

    return reachable_edges


def edge_arrays(edge_type, mu_mn=0.0, mu_h2o=0.0):
    x, y = [], []

    reachable_edges = get_reachable_edges(
    mu_mn,
    mu_h2o
)

    for a_name, b_name, transition in reachable_edges:
        if transition != edge_type:
            continue

        a = smap[a_name]
        b = smap[b_name]

        Ea = corrected_energy(a, mu_mn, mu_h2o)
        Eb = corrected_energy(b, mu_mn, mu_h2o)

        dE = Eb - Ea

        if dE > BARRIER_THRESHOLD_EV:
            continue

        x += [
            a["coordinate"] + 0.18,
            b["coordinate"] - 0.18,
            None
        ]
        y += [Ea, Eb, None]

    return x, y


In [6]:
# ============================================================
# 4. Plot
# ============================================================
fig = go.FigureWidget()

for transition in ["OL", "C", "SR", "OX"]:
    x, y = edge_arrays(transition)

    fig.add_scatter(
        x=x,
        y=y,
        mode="lines",
        line=dict(width=2.2, color=COLOR[transition]),
        opacity=0.72,
        name=transition,
        hoverinfo="skip",
    )


# Short horizontal energy levels
level_x = []
level_y = []

marker_x = []
marker_y = []
labels = []
custom = []

for s in data:
    y = corrected_energy(s, 0.0, 0.0)

    level_x += [
        s["coordinate"] - 0.18,
        s["coordinate"] + 0.18,
        None,
    ]
    level_y += [y, y, None]

    marker_x.append(s["coordinate"])
    marker_y.append(y)
    labels.append(s["state"])

    custom.append(
        [
            s["state"],
            s["path"],
            s["E0"],
            s["O"],
            s["C"],
            s["X"],
        ]
    )


fig.add_scatter(
    x=level_x,
    y=level_y,
    mode="lines",
    line=dict(width=3.5, color="#111111"),
    hoverinfo="skip",
    showlegend=False,
)

fig.add_scatter(
    x=marker_x,
    y=marker_y,
    mode="markers+text",
    text=labels,
    textposition="top center",
    marker=dict(size=8, color="#111111"),
    customdata=custom,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Path: %{customdata[1]}<br>"
        "Reaction coordinate: %{x}<br>"
        "Corrected relative energy: %{y:.6f} eV<br>"
        "E0: %{customdata[2]:.6f} eV<br>"
        "O=%{customdata[3]}, "
        "C=%{customdata[4]}, "
        "X=%{customdata[5]}"
        "<extra></extra>"
    ),
    showlegend=False,
)


PLOT_WIDTH = 1200
NETWORK_PLOT_HEIGHT = 800
BEST_PLOT_HEIGHT = 700
PLOT_MARGIN = dict(l=85, r=40, t=135, b=75)
MU_MN_LIMITS = (-4.0, 0.0)
MU_H2O_LIMITS = (-4.0, 0.0)

fixed_y_values = [
    corrected_energy(s, mu_mn_value, mu_h2o_value)
    for s in data
    for mu_mn_value in MU_MN_LIMITS
    for mu_h2o_value in MU_H2O_LIMITS
]
FIXED_Y_RANGE = [
    min(fixed_y_values) - 0.5,
    max(fixed_y_values) + 0.5,
]

fig.update_layout(
    title="Chemical-potential dependent free-energy diagram",
    xaxis_title="Reaction coordinate",
    yaxis_title="Relative energy (eV)",
    xaxis=dict(
        dtick=1,
        range=[-0.45, 7.45],
        fixedrange=False,
    ),
    yaxis=dict(
        range=FIXED_Y_RANGE,
        autorange=False,
        fixedrange=False,
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.10,
    ),
    template="plotly_white",
    autosize=False,
    width=PLOT_WIDTH,
    height=NETWORK_PLOT_HEIGHT,
    margin=PLOT_MARGIN,
)

FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': '#2563eb', 'width': 2.2},
              'mode': 'lines',
              'name': 'OL',
              'opacity': 0.72,
              'type': 'scatter',
              'uid': '4b8a7e05-f0d1-4399-a395-b37c41d4af1f',
              'x': [0.18, 0.8200000000000001, None, 1.18, 1.82, None, 2.18, 2.82,
                    None, 1.18, 1.82, None, 2.18, 2.82, None, 3.18, 3.82, None,
                    3.18, 3.82, None, 4.18, 4.82, None, 3.18, 3.82, None, 5.18,
                    5.82, None, 4.18, 4.82, None, 5.18, 5.82, None, 6.18, 6.82,
                    None],
              'y': [0.0, 0.1403215, None, -0.717431781, 0.045049719, None,
                    -0.673013563, -0.571462062, None, 0.1403215, 0.741473, None,
                    0.045049719, 0.624151219, None, -0.571462062, -1.352820562,
                    None, 0.321683469, -0.712705031, None, -0.785008313,
                    -1.441116812, None, 0.62415121

In [11]:

# ============================================================
# 5. Interactive chemical-potential sliders
# ============================================================
mu_mn = widgets.FloatSlider(
    value=0.0,
    min=-4.0,
    max=0.0,
    step=0.05,
    description="Δμ Mn(OH)2",
    continuous_update=True,
    readout_format=".2f",
    layout=widgets.Layout(width="650px"),
)

mu_h2o = widgets.FloatSlider(
    value=0.0,
    min=-4.0,
    max=0.0,
    step=0.05,
    description="Δμ H2O",
    continuous_update=True,
    readout_format=".2f",
    layout=widgets.Layout(width="650px"),
)


def update(change=None):
    with fig.batch_update():

        # Update reaction-path lines.
        for i, transition in enumerate(["OL", "C", "SR", "OX"]):
            x, y = edge_arrays(
                transition,
                mu_mn.value,
                mu_h2o.value,
            )

            fig.data[i].x = x
            fig.data[i].y = y

        # Update state energy levels and markers.
        level_x = []
        level_y = []
        marker_y = []

        for s in data:
            y = corrected_energy(
                s,
                mu_mn.value,
                mu_h2o.value,
            )

            level_x += [
                s["coordinate"] - 0.18,
                s["coordinate"] + 0.18,
                None,
            ]
            level_y += [y, y, None]
            marker_y.append(y)

        fig.data[4].x = level_x
        fig.data[4].y = level_y
        fig.data[5].y = marker_y

        fig.layout.title = (
            "Chemical-potential dependent free-energy diagram"
            f"<br><sup>"
            f"ΔμMn(OH)2 = {mu_mn.value:.2f} eV, "
            f"ΔμH2O = {mu_h2o.value:.2f} eV"
            f"</sup>"
        )

        fig.layout.yaxis.autorange = False

mu_mn.observe(update, names="value")
mu_h2o.observe(update, names="value")


print(f"Loaded {len(data)} states from: {CSV_FILE}")
print(f"Generated {len(edges)} reaction-network edges.")
display(widgets.VBox([mu_mn, mu_h2o]), fig)

Loaded 27 states from: /home/sowon-desktop/ligand-project/EnergeticSpanModel/rNets_diagram/plotly/energy_011_12.csv
Generated 66 reaction-network edges.


FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': '#2563eb', 'width': 2.2},
              'mode': 'lines',
              'name': 'OL',
              'opacity': 0.72,
              'type': 'scatter',
              'uid': '4b8a7e05-f0d1-4399-a395-b37c41d4af1f',
              'x': [0.18, 0.8200000000000001, None, 1.18, 1.82, None, 2.18, 2.82,
                    None, 1.18, 1.82, None, 2.18, 2.82, None, 3.18, 3.82, None,
                    3.18, 3.82, None, 4.18, 4.82, None, 3.18, 3.82, None, 5.18,
                    5.82, None, 4.18, 4.82, None, 5.18, 5.82, None, 6.18, 6.82,
                    None],
              'y': [0.0, 0.1403215, None, -0.717431781, 0.045049719, None,
                    -0.673013563, -0.571462062, None, 0.1403215, 0.741473, None,
                    0.045049719, 0.624151219, None, -0.571462062, -1.352820562,
                    None, 0.321683469, -0.712705031, None, -0.785008313,
                    -1.441116812, None, 0.62415121

In [8]:
# ============================================================
# 6. Plot ONLY the minimum-δE pathway
# ============================================================
from collections import defaultdict
import numpy as np

START_STATE = "S000"
FINAL_STATE = "S322"

TIE_TOLERANCE_EV = 1e-6
LEVEL_HALF_WIDTH = 0.18

# True  = δE가 정확히 같은 co-optimal path를 모두 표시
# False = 그중 첫 번째 representative path 하나만 표시
SHOW_ALL_COOPTIMAL = False


# ------------------------------------------------------------
# 1. 모든 complete path 생성
#    여기에는 ΔG < 0.8 eV 등의 필터를 적용하지 않음
# ------------------------------------------------------------
edge_type_map = {
    (a, b): transition
    for a, b, transition in edges
}

adjacency = defaultdict(list)

for a, b, transition in edges:
    adjacency[a].append(b)

for node in adjacency:
    adjacency[node].sort()


raw_paths = []

def dfs_all_paths(node, current_path):

    if node == FINAL_STATE:
        raw_paths.append(current_path.copy())
        return

    for next_node in adjacency.get(node, []):

        if next_node in current_path:
            continue

        dfs_all_paths(
            next_node,
            current_path + [next_node]
        )


dfs_all_paths(
    START_STATE,
    [START_STATE]
)

raw_paths.sort(
    key=lambda p: " -> ".join(p)
)


all_paths = []

for i, path in enumerate(raw_paths, start=1):

    mechanisms = [
        edge_type_map[(a, b)]
        for a, b in zip(
            path[:-1],
            path[1:]
        )
    ]

    all_paths.append(
        {
            "path_id": f"P{i:03d}",
            "states": path,
            "mechanisms": mechanisms,
        }
    )


print(
    f"Complete unfiltered paths: {len(all_paths)}"
)


# ------------------------------------------------------------
# 2. 경로 하나의 energetic span 계산
# ------------------------------------------------------------
def energetic_span_of_path(
    path_info,
    mu_mn_value,
    mu_h2o_value,
):

    names = path_info["states"]

    energies = [
        corrected_energy(
            smap[name],
            mu_mn_value,
            mu_h2o_value,
        )
        for name in names
    ]

    # 전체 reaction free energy
    dGr = energies[-1] - energies[0]

    # final state는 TDI/TDTS 후보에서 제외
    candidate_names = names[:-1]
    candidate_E = energies[:-1]

    max_span = -np.inf
    max_pairs = []

    for i, Gi in enumerate(candidate_E):

        for j, Gj in enumerate(candidate_E):

            # 기존에 사용한 energetic span convention
            correction = dGr if i < j else 0.0

            span = (
                Gi
                - Gj
                + correction
            )

            if span > max_span + TIE_TOLERANCE_EV:

                max_span = span

                max_pairs = [
                    (
                        candidate_names[j],  # TDI
                        candidate_names[i],  # TDTS
                    )
                ]

            elif abs(
                span - max_span
            ) <= TIE_TOLERANCE_EV:

                max_pairs.append(
                    (
                        candidate_names[j],
                        candidate_names[i],
                    )
                )

    return {
        "path_id": path_info["path_id"],
        "states": path_info["states"],
        "mechanisms": path_info["mechanisms"],
        "deltaE": max_span,
        "dGr": dGr,
        "pairs": max_pairs,
    }


# ------------------------------------------------------------
# 3. 모든 경로 계산 → minimum δE path 선택
# ------------------------------------------------------------
def find_minimum_deltaE_paths(
    mu_mn_value,
    mu_h2o_value,
):

    results = [
        energetic_span_of_path(
            path,
            mu_mn_value,
            mu_h2o_value,
        )
        for path in all_paths
    ]

    min_deltaE = min(
        r["deltaE"]
        for r in results
    )

    best_paths = [
        r
        for r in results
        if abs(
            r["deltaE"] - min_deltaE
        ) <= TIE_TOLERANCE_EV
    ]

    best_paths.sort(
        key=lambda r: r["path_id"]
    )

    return min_deltaE, best_paths


# ------------------------------------------------------------
# 4. Plotly figure 생성
# ------------------------------------------------------------
fig_best = go.FigureWidget()


# transition lines
for transition in ["OL", "C", "SR", "OX"]:

    fig_best.add_scatter(
        x=[],
        y=[],
        mode="lines",
        line=dict(
            width=4,
            color=COLOR[transition],
        ),
        name=transition,
        hoverinfo="skip",
    )


# horizontal state energy levels
fig_best.add_scatter(
    x=[],
    y=[],
    mode="lines",
    line=dict(
        width=4,
        color="black",
    ),
    hoverinfo="skip",
    showlegend=False,
)


# state markers + labels
fig_best.add_scatter(
    x=[],
    y=[],
    mode="markers+text",
    text=[],
    textposition="top center",
    marker=dict(
        size=9,
        color="black",
    ),
    showlegend=False,
)


# TDI marker
fig_best.add_scatter(
    x=[],
    y=[],
    mode="markers+text",
    text=[],
    textposition="bottom center",
    marker=dict(
        size=18,
        symbol="star",
        color="#00AEEF",
        line=dict(
            width=1.5,
            color="black",
        ),
    ),
    name="TDI",
)


# TDTS marker
fig_best.add_scatter(
    x=[],
    y=[],
    mode="markers+text",
    text=[],
    textposition="bottom center",
    marker=dict(
        size=15,
        symbol="diamond",
        color="#E53935",
        line=dict(
            width=1.5,
            color="black",
        ),
    ),
    name="TDTS",
)


fig_best.update_layout(

    title="Minimum energetic-span pathway",

    xaxis_title="Reaction coordinate",
    yaxis_title="Relative energy (eV)",

    xaxis=dict(
        dtick=1,
        range=[-0.45, 7.45],
        fixedrange=False,
    ),
    yaxis=dict(
        range=FIXED_Y_RANGE,
        autorange=False,
        fixedrange=False,
    ),

    template="plotly_white",

    autosize=False,
    width=PLOT_WIDTH,
    height=BEST_PLOT_HEIGHT,
    margin=PLOT_MARGIN,

    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.12,
    ),
)


# ------------------------------------------------------------
# 5. 현재 chemical potential에서 best path 그림 생성
# ------------------------------------------------------------
def update_best_path(change=None):

    muMn = mu_mn.value
    muH2O = mu_h2o.value

    min_deltaE, best_paths = (
        find_minimum_deltaE_paths(
            muMn,
            muH2O,
        )
    )

    if SHOW_ALL_COOPTIMAL:

        selected_paths = best_paths

    else:

        # 여러 경로가 tie이면 representative 하나만 표시
        selected_paths = [
            best_paths[0]
        ]


    # ========================================================
    # 사용되는 edge / state 모으기
    # ========================================================
    selected_edges = {
        transition: set()
        for transition in [
            "OL",
            "C",
            "SR",
            "OX",
        ]
    }

    selected_states = set()

    tdi_states = set()
    tdts_states = set()


    for result in selected_paths:

        path = result["states"]
        mechanisms = result["mechanisms"]

        selected_states.update(path)

        for a, b, mechanism in zip(
            path[:-1],
            path[1:],
            mechanisms,
        ):

            selected_edges[
                mechanism
            ].add(
                (a, b)
            )


        for TDI, TDTS in result["pairs"]:

            tdi_states.add(TDI)
            tdts_states.add(TDTS)


    # ========================================================
    # transition lines
    # ========================================================
    with fig_best.batch_update():

        for trace_index, transition in enumerate(
            ["OL", "C", "SR", "OX"]
        ):

            x = []
            y = []

            for a_name, b_name in sorted(
                selected_edges[transition]
            ):

                a = smap[a_name]
                b = smap[b_name]

                Ea = corrected_energy(
                    a,
                    muMn,
                    muH2O,
                )

                Eb = corrected_energy(
                    b,
                    muMn,
                    muH2O,
                )

                # horizontal level 끝 ↔ 끝 연결
                x += [
                    a["coordinate"]
                    + LEVEL_HALF_WIDTH,

                    b["coordinate"]
                    - LEVEL_HALF_WIDTH,

                    None,
                ]

                y += [
                    Ea,
                    Eb,
                    None,
                ]


            fig_best.data[
                trace_index
            ].x = x

            fig_best.data[
                trace_index
            ].y = y


        # ====================================================
        # horizontal energy levels
        # ====================================================
        level_x = []
        level_y = []

        marker_x = []
        marker_y = []
        marker_text = []


        sorted_states = sorted(
            selected_states,
            key=lambda name: (
                smap[name]["coordinate"],
                name,
            ),
        )


        for name in sorted_states:

            s = smap[name]

            E = corrected_energy(
                s,
                muMn,
                muH2O,
            )

            level_x += [
                s["coordinate"]
                - LEVEL_HALF_WIDTH,

                s["coordinate"]
                + LEVEL_HALF_WIDTH,

                None,
            ]

            level_y += [
                E,
                E,
                None,
            ]

            marker_x.append(
                s["coordinate"]
            )

            marker_y.append(E)

            marker_text.append(name)


        fig_best.data[4].x = level_x
        fig_best.data[4].y = level_y

        fig_best.data[5].x = marker_x
        fig_best.data[5].y = marker_y
        fig_best.data[5].text = marker_text


        # ====================================================
        # TDI
        # ====================================================
        tdi_sorted = sorted(
            tdi_states,
            key=lambda name: (
                smap[name]["coordinate"],
                name,
            )
        )

        fig_best.data[6].x = [
            smap[name]["coordinate"]
            for name in tdi_sorted
        ]

        fig_best.data[6].y = [
            corrected_energy(
                smap[name],
                muMn,
                muH2O,
            )
            for name in tdi_sorted
        ]

        fig_best.data[6].text = [
            f"TDI: {name}"
            for name in tdi_sorted
        ]


        # ====================================================
        # TDTS
        # ====================================================
        tdts_sorted = sorted(
            tdts_states,
            key=lambda name: (
                smap[name]["coordinate"],
                name,
            )
        )

        fig_best.data[7].x = [
            smap[name]["coordinate"]
            for name in tdts_sorted
        ]

        fig_best.data[7].y = [
            corrected_energy(
                smap[name],
                muMn,
                muH2O,
            )
            for name in tdts_sorted
        ]

        fig_best.data[7].text = [
            f"TDTS: {name}"
            for name in tdts_sorted
        ]


        # ====================================================
        # title
        # ====================================================
        path_ids = [
            r["path_id"]
            for r in best_paths
        ]


        if len(path_ids) == 1:

            path_text = path_ids[0]

        else:

            path_text = (
                f"{path_ids[0]} "
                f"(+ {len(path_ids)-1} co-optimal)"
            )


        representative = best_paths[0]

        sequence = " → ".join(
            representative["states"]
        )


        fig_best.layout.title = (

            "<b>Minimum energetic-span pathway</b>"

            f"<br><sup>"
            f"ΔμMn(OH)₂ = {muMn:.2f} eV, "
            f"ΔμH₂O = {muH2O:.2f} eV"

            f" | δEmin = "
            f"{min_deltaE:.4f} eV"

            f" | {path_text}"

            f"<br>{sequence}"

            f"</sup>"
        )


        fig_best.layout.yaxis.autorange = False


# ------------------------------------------------------------
# 6. 기존 chemical-potential slider와 연결
# ------------------------------------------------------------
mu_mn.observe(
    update_best_path,
    names="value",
)

mu_h2o.observe(
    update_best_path,
    names="value",
)


# initial drawing
update_best_path()


display(fig_best)

Complete unfiltered paths: 255


FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': '#2563eb', 'width': 4},
              'mode': 'lines',
              'name': 'OL',
              'type': 'scatter',
              'uid': 'c653f34a-1961-4815-bfcc-bd74f474c357',
              'x': [4.18, 4.82, None],
              'y': [-0.785008313, -1.441116812, None]},
             {'hoverinfo': 'skip',
              'line': {'color': '#9333ea', 'width': 4},
              'mode': 'lines',
              'name': 'C',
              'type': 'scatter',
              'uid': 'fcefd88f-025a-452b-933b-233764b681fd',
              'x': [],
              'y': []},
             {'hoverinfo': 'skip',
              'line': {'color': '#16a34a', 'width': 4},
              'mode': 'lines',
              'name': 'SR',
              'type': 'scatter',
              'uid': 'e00c0020-f1fc-4e01-b0c4-3b435dc3096f',
              'x': [0.18, 0.8200000000000001, None, 1.18, 1.82, None],
              'y': [0.0, -0.717431781, No

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ============================================================
# Chemical potential sliders
# ============================================================

mu_mn = widgets.FloatSlider(
    value=0.0,
    min=-4.0,
    max=0.0,
    step=0.05,
    description="Δμ Mn(OH)₂",
    continuous_update=True,
    readout=True,
    readout_format=".2f",
    layout=widgets.Layout(width="700px"),
)

mu_h2o = widgets.FloatSlider(
    value=0.0,
    min=-4.0,
    max=0.0,
    step=0.05,
    description="Δμ H₂O",
    continuous_update=True,
    readout=True,
    readout_format=".2f",
    layout=widgets.Layout(width="700px"),
)


# ============================================================
# Slider → minimum δE pathway update
# ============================================================

mu_mn.observe(
    update_best_path,
    names="value"
)

mu_h2o.observe(
    update_best_path,
    names="value"
)


# initial calculation
update_best_path()


# ============================================================
# Display
# ============================================================

controls = widgets.VBox([
    mu_mn,
    mu_h2o,
])

display(
    controls,
    fig_best
)

from plotly_standalone_export import (
    write_best_path_html,
    write_network_html,
)

write_network_html(
    fig,
    BASE_DIR / "plotly_diagram_011_all_paths.html",
    data,
    edges,
    BARRIER_THRESHOLD_EV,
)
write_best_path_html(
    fig_best,
    BASE_DIR / "plotly_diagram_011_minimum_path.html",
    data,
    all_paths,
    TIE_TOLERANCE_EV,
    LEVEL_HALF_WIDTH,
    SHOW_ALL_COOPTIMAL,
)

FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': '#2563eb', 'width': 4},
              'mode': 'lines',
              'name': 'OL',
              'type': 'scatter',
              'uid': 'c653f34a-1961-4815-bfcc-bd74f474c357',
              'x': [4.18, 4.82, None],
              'y': [-0.785008313, -1.441116812, None]},
             {'hoverinfo': 'skip',
              'line': {'color': '#9333ea', 'width': 4},
              'mode': 'lines',
              'name': 'C',
              'type': 'scatter',
              'uid': 'fcefd88f-025a-452b-933b-233764b681fd',
              'x': [],
              'y': []},
             {'hoverinfo': 'skip',
              'line': {'color': '#16a34a', 'width': 4},
              'mode': 'lines',
              'name': 'SR',
              'type': 'scatter',
              'uid': 'e00c0020-f1fc-4e01-b0c4-3b435dc3096f',
              'x': [0.18, 0.8200000000000001, None, 1.18, 1.82, None],
              'y': [0.0, -0.717431781, No